In [0]:
%pip install xgboost scikit-learn
dbutils.library.restartPython()

# 🎯 Teste Final de Holdout - Modelo Semanal FII

## 📊 Objetivo

Executar a **avaliação final e única** do modelo semanal de 7 pregões no período completamente intocado: **01/03/2025 a 28/02/2026**.

Esta é a **execução final**. Nenhuma modificação será feita no modelo após observar os resultados.

---

## 🔒 Metodologia Congelada

### Dados
* **Tabela**: workspace.gold.fii_features_v1
* **Features**: Exatamente as mesmas 23 features do modelo aprovado
* **Target**: target_7d (binário: FII > IFIX em 7 pregões)

### Modelo
* **Algoritmo**: XGBoost ajustado
* **Hiperparâmetros**: FINAL_HYPERPARAMETERS_REVISED do notebook 38_hyperparameter_tuning
* **Seed**: 42 (mesma utilizada anteriormente)

### Restrições
* ❌ Nenhuma nova feature
* ❌ Nenhum tuning
* ❌ Nenhuma seleção de features
* ❌ Nenhuma alteração de threshold
* ❌ Nenhuma modificação após observar resultados

---

## ⚠️ Purge Gap de 7 Pregões

**IMPORTANTE**: Aplicar purge gap de 7 pregões antes do início do holdout.

Não basta filtrar treino por `date < 01/03/2025`, pois os targets das últimas linhas de fevereiro podem utilizar retornos pertencentes a março de 2025.

**Determinar a última data de treino cujo horizonte completo de target_7d termina antes de 01/03/2025.**

---

## 📈 Avaliação

### Métricas Gerais
* ROC-AUC
* Accuracy, Precision, Recall, F1
* Matriz de confusão
* Distribuição de probabilidades
* Calibração básica

### Métricas por Segmento
* Taxa de acerto por ticker
* ROC-AUC por ticker (quando viável)
* Taxa de acerto por mês
* ROC-AUC por mês (quando houver ambas as classes)
* Estabilidade ao longo dos 12 meses

### Avaliação Top 1
Para cada data do holdout:
1. Ranquear os 5 FIIs pela probabilidade prevista
2. Selecionar o FII ranking 1
3. Verificar se superou o IFIX nos próximos 7 pregões
4. Calcular taxa Top 1 de outperformance
5. Calcular quantas vezes o Top 1 foi o melhor retorno absoluto

**Meta aproximada**: 58% (sem alterar regras para alcançá-la)

In [0]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Seed para reprodutibilidade (mesma do modelo)
SEED = 42
np.random.seed(SEED)

print("=" * 80)
print("✅ TESTE FINAL DE HOLDOUT - MODELO SEMANAL FII")
print("=" * 80)
print(f"\nData de execução: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Seed: {SEED}")
print("\n⚠️  Esta é a execução final. Nenhuma modificação será feita após os resultados.")
print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("🔧 HIPERPARÂMETROS DO MODELO CONGELADO")
print("=" * 80)

# Hiperparâmetros finais do notebook 38_hyperparameter_tuning (FINAL_HYPERPARAMETERS_REVISED)
FINAL_HYPERPARAMETERS = {
    'n_estimators': 150,
    'max_depth': 6,
    'learning_rate': 0.07,
    'min_child_weight': 1,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'gamma': 0.0,
    'reg_alpha': 0.01,
    'reg_lambda': 1.5,
    'early_stopping_rounds': 20,
    'random_state': SEED,
    'eval_metric': 'auc',
    'verbosity': 0
}

print("\nConfiguração XGBoost AJUSTADO (do notebook 38):")
for k, v in FINAL_HYPERPARAMETERS.items():
    print(f"  {k}: {v}")

print("\n✅ Hiperparâmetros carregados e congelados")
print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("📋 CARREGANDO DADOS GOLD V1")
print("=" * 80)

# Carregar Gold V1 reconstruída
df = spark.table("workspace.gold.fii_features_v1").toPandas()

print(f"\n✅ Gold V1 carregada: {df.shape[0]:,} registros, {df.shape[1]} colunas")

# Converter date
df['date'] = pd.to_datetime(df['date'])

# Ordenar
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

print(f"Período: {df['date'].min().date()} a {df['date'].max().date()}")
print(f"Tickers: {df['ticker'].nunique()} ({', '.join(sorted(df['ticker'].unique()))})")

# Colunas não-features
non_feature_cols = ['ticker', 'date', 'target_7d', 'target_alpha_7d']

# Features (exatamente as mesmas 23 features)
features = [col for col in df.columns if col not in non_feature_cols]

print(f"\nFeatures: {len(features)}")
for i, f in enumerate(features, 1):
    print(f"  {i:2d}. {f}")

print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("⚠️  DETERMINAR PURGE GAP DE 7 PREGÕES")
print("=" * 80)

# Período de holdout: 01/03/2025 a 28/02/2026
holdout_start = pd.to_datetime('2025-03-01')
holdout_end = pd.to_datetime('2026-02-28')

print(f"\nPeríodo de holdout definido: {holdout_start.date()} a {holdout_end.date()}")

# Encontrar todas as datas antes do holdout
dates_before_holdout = df[df['date'] < holdout_start]['date'].unique()
dates_before_holdout = sorted(dates_before_holdout)

print(f"\nDatas disponíveis antes do holdout: {len(dates_before_holdout)}")
print(f"\u00daltimas 10 datas antes do holdout:")
for d in dates_before_holdout[-10:]:
    print(f"  {pd.to_datetime(d).date()}")

# Determinar a última data de treino (7 pregões antes do holdout)
# O target_7d usa os próximos 7 pregões, então precisamos garantir que
# nenhum target de treino alcance março de 2025
if len(dates_before_holdout) >= 7:
    # A última data segura de treino é a 7ª data antes do início do holdout
    last_train_date = dates_before_holdout[-7]
    purged_dates = dates_before_holdout[-7:]
else:
    raise ValueError("Não há datas suficientes antes do holdout para aplicar purge gap!")

print(f"\n✅ Última data de treino (antes do purge gap): {pd.to_datetime(last_train_date).date()}")
print(f"\n🚫 Datas removidas pelo purge gap de 7 pregões:")
for i, d in enumerate(purged_dates, 1):
    print(f"  {i}. {pd.to_datetime(d).date()}")

# Primeira data do holdout
first_holdout_date = df[df['date'] >= holdout_start]['date'].min()
print(f"\n🎯 Primeira data do holdout: {pd.to_datetime(first_holdout_date).date()}")

print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("✂️ DIVIDIR TREINO / HOLDOUT")
print("=" * 80)

# Treino: até a última data segura (antes do purge gap)
train_df = df[df['date'] <= last_train_date].copy()

# Holdout: período intocado (01/03/2025 a 28/02/2026)
holdout_df = df[(df['date'] >= holdout_start) & (df['date'] <= holdout_end)].copy()

print(f"\n📋 Treino:")
print(f"  Período: {train_df['date'].min().date()} a {train_df['date'].max().date()}")
print(f"  Registros: {len(train_df):,}")
print(f"  Registros por ticker:")
for ticker in sorted(train_df['ticker'].unique()):
    count = len(train_df[train_df['ticker'] == ticker])
    print(f"    {ticker}: {count:,}")

print(f"\n🎯 Holdout:")
print(f"  Período: {holdout_df['date'].min().date()} a {holdout_df['date'].max().date()}")
print(f"  Registros: {len(holdout_df):,}")
print(f"  Registros por ticker:")
for ticker in sorted(holdout_df['ticker'].unique()):
    count = len(holdout_df[holdout_df['ticker'] == ticker])
    print(f"    {ticker}: {count:,}")

# Distribuição do target
print(f"\n📊 Distribuição do target:")
print(f"\n  Treino:")
train_target_dist = train_df['target_7d'].value_counts()
for k, v in train_target_dist.items():
    pct = v / len(train_df) * 100
    print(f"    {k}: {v:,} ({pct:.1f}%)")

print(f"\n  Holdout:")
holdout_target_dist = holdout_df['target_7d'].value_counts()
for k, v in holdout_target_dist.items():
    pct = v / len(holdout_df) * 100
    print(f"    {k}: {v:,} ({pct:.1f}%)")

print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("✅ VALIDAÇÕES CRÍTICAS")
print("=" * 80)

# 1. Confirmar ausência de sobreposição temporal
max_train_date = train_df['date'].max()
min_holdout_date = holdout_df['date'].min()

print(f"\n1️⃣ Ausência de sobreposição temporal:")
print(f"  Última data treino: {max_train_date.date()}")
print(f"  Primeira data holdout: {min_holdout_date.date()}")
print(f"  Gap temporal: {(min_holdout_date - max_train_date).days} dias")
if max_train_date < min_holdout_date:
    print(f"  ✅ SEM sobreposição")
else:
    print(f"  ❌ ERRO: Há sobreposição!")

# 2. Confirmar que nenhum target de treino alcança março de 2025
print(f"\n2️⃣ Confirmar que nenhum target de treino usa dados do holdout:")

# Para cada data de treino, verificar se 7 pregões à frente cairiam no holdout
train_dates_sorted = sorted(train_df['date'].unique())
all_dates_sorted = sorted(df['date'].unique())

# Para a última data de treino, encontrar qual seria a 7ª data futura
last_train_idx = all_dates_sorted.index(max_train_date)
if last_train_idx + 7 < len(all_dates_sorted):
    date_7_forward = all_dates_sorted[last_train_idx + 7]
    print(f"  Última data treino: {max_train_date.date()}")
    print(f"  7 pregões à frente: {date_7_forward.date()}")
    print(f"  Início holdout: {holdout_start.date()}")
    if date_7_forward < holdout_start:
        print(f"  ✅ Target de treino NÃO alcança o holdout")
    else:
        print(f"  ❌ ERRO: Target de treino alcança o holdout!")
else:
    print(f"  ⚠️ Não há 7 pregões suficientes após a última data de treino")

# 3. Verificar nulos
print(f"\n3️⃣ Verificar nulos nas features:")
train_nulls = train_df[features].isnull().sum()
holdout_nulls = holdout_df[features].isnull().sum()

if train_nulls.sum() > 0:
    print(f"  ⚠️ Treino tem nulos:")
    for col in train_nulls[train_nulls > 0].index:
        print(f"    {col}: {train_nulls[col]} nulos")
else:
    print(f"  ✅ Treino: sem nulos")

if holdout_nulls.sum() > 0:
    print(f"  ⚠️ Holdout tem nulos:")
    for col in holdout_nulls[holdout_nulls > 0].index:
        print(f"    {col}: {holdout_nulls[col]} nulos")
else:
    print(f"  ✅ Holdout: sem nulos")

print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("📊 PREPARAR DADOS (X, y)")
print("=" * 80)

# Separar features e target
X_train = train_df[features].values
y_train = train_df['target_7d'].values

X_holdout = holdout_df[features].values
y_holdout = holdout_df['target_7d'].values

print(f"\nTreino:")
print(f"  X_train: {X_train.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  Distribuição y: {np.bincount(y_train.astype(int))}")

print(f"\nHoldout:")
print(f"  X_holdout: {X_holdout.shape}")
print(f"  y_holdout: {y_holdout.shape}")
print(f"  Distribuição y: {np.bincount(y_holdout.astype(int))}")

print("\n✅ Dados preparados")
print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("🎯 TREINAR MODELO XGBOOST (UMA Única VEZ)")
print("=" * 80)

print("\n⚠️  Treinando modelo com hiperparâmetros congelados...")
print("\nEsta é a execução final. O modelo NÃO será modificado após observar os resultados.\n")

# Criar modelo
model = xgb.XGBClassifier(**FINAL_HYPERPARAMETERS)

# Treinar
import time
start_time = time.time()

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train)],
    verbose=False
)

training_time = time.time() - start_time

print(f"\n✅ Modelo treinado em {training_time:.2f} segundos")
print(f"\nNúmero de árvores: {model.n_estimators}")
print(f"Profundidade máxima: {model.max_depth}")
print(f"Taxa de aprendizado: {model.learning_rate}")

print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("🔮 PREDIÇÕES NO HOLDOUT")
print("=" * 80)

# Predições de probabilidade
y_pred_proba = model.predict_proba(X_holdout)[:, 1]

# Predições binárias (threshold 0.5)
y_pred = (y_pred_proba >= 0.5).astype(int)

print(f"\nPredições realizadas: {len(y_pred):,}")
print(f"\nDistribuição de probabilidades:")
print(f"  Mínimo: {y_pred_proba.min():.4f}")
print(f"  Máximo: {y_pred_proba.max():.4f}")
print(f"  Média: {y_pred_proba.mean():.4f}")
print(f"  Mediana: {np.median(y_pred_proba):.4f}")
print(f"  Desvio padrão: {y_pred_proba.std():.4f}")

print(f"\nPreviões binárias (threshold 0.5):")
print(f"  Classe 0: {(y_pred == 0).sum():,}")
print(f"  Classe 1: {(y_pred == 1).sum():,}")

print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("📊 MÉTRICAS GERAIS NO HOLDOUT")
print("=" * 80)

# ROC-AUC
roc_auc = roc_auc_score(y_holdout, y_pred_proba)

# Outras métricas
accuracy = accuracy_score(y_holdout, y_pred)
precision = precision_score(y_holdout, y_pred, zero_division=0)
recall = recall_score(y_holdout, y_pred, zero_division=0)
f1 = f1_score(y_holdout, y_pred, zero_division=0)

print(f"\n🎯 Métricas Principais:")
print(f"  ROC-AUC: {roc_auc:.4f}")
print(f"  Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1-Score: {f1:.4f}")

# Matriz de confusão
cm = confusion_matrix(y_holdout, y_pred)

print(f"\n📊 Matriz de Confusão:")
print(f"\n  Real \ Previsto    0         1")
print(f"  0              {cm[0,0]:5d}     {cm[0,1]:5d}")
print(f"  1              {cm[1,0]:5d}     {cm[1,1]:5d}")

tn, fp, fn, tp = cm.ravel()
print(f"\n  Verdadeiros Negativos (TN): {tn}")
print(f"  Falsos Positivos (FP): {fp}")
print(f"  Falsos Negativos (FN): {fn}")
print(f"  Verdadeiros Positivos (TP): {tp}")

print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("🎯 ANÁLISE POR TICKER")
print("=" * 80)

# Adicionar predições ao holdout_df
holdout_df_eval = holdout_df.copy()
holdout_df_eval['y_pred'] = y_pred
holdout_df_eval['y_pred_proba'] = y_pred_proba

print(f"\n📊 Métricas por Ticker:\n")

ticker_metrics = []

for ticker in sorted(holdout_df_eval['ticker'].unique()):
    ticker_data = holdout_df_eval[holdout_df_eval['ticker'] == ticker]
    
    y_true_ticker = ticker_data['target_7d'].values
    y_pred_ticker = ticker_data['y_pred'].values
    y_proba_ticker = ticker_data['y_pred_proba'].values
    
    # Métricas
    acc = accuracy_score(y_true_ticker, y_pred_ticker)
    
    # ROC-AUC (apenas se ambas as classes estão presentes)
    if len(np.unique(y_true_ticker)) > 1:
        auc_ticker = roc_auc_score(y_true_ticker, y_proba_ticker)
    else:
        auc_ticker = np.nan
    
    ticker_metrics.append({
        'ticker': ticker,
        'n_obs': len(ticker_data),
        'accuracy': acc,
        'roc_auc': auc_ticker
    })
    
    print(f"{ticker}:")
    print(f"  Observações: {len(ticker_data):,}")
    print(f"  Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    if not np.isnan(auc_ticker):
        print(f"  ROC-AUC: {auc_ticker:.4f}")
    else:
        print(f"  ROC-AUC: N/A (apenas uma classe presente)")
    print()

# Resumo
ticker_metrics_df = pd.DataFrame(ticker_metrics)

print("\n📈 Resumo por Ticker:")
print(f"\nMelhor Accuracy: {ticker_metrics_df.loc[ticker_metrics_df['accuracy'].idxmax(), 'ticker']} ({ticker_metrics_df['accuracy'].max():.4f})")
print(f"Pior Accuracy: {ticker_metrics_df.loc[ticker_metrics_df['accuracy'].idxmin(), 'ticker']} ({ticker_metrics_df['accuracy'].min():.4f})")

if ticker_metrics_df['roc_auc'].notna().any():
    valid_auc = ticker_metrics_df[ticker_metrics_df['roc_auc'].notna()]
    print(f"\nMelhor ROC-AUC: {valid_auc.loc[valid_auc['roc_auc'].idxmax(), 'ticker']} ({valid_auc['roc_auc'].max():.4f})")
    print(f"Pior ROC-AUC: {valid_auc.loc[valid_auc['roc_auc'].idxmin(), 'ticker']} ({valid_auc['roc_auc'].min():.4f})")

print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("📅 ANÁLISE POR MÊS")
print("=" * 80)

# Adicionar mês
holdout_df_eval['year_month'] = holdout_df_eval['date'].dt.to_period('M')

print(f"\n📊 Métricas por Mês:\n")

month_metrics = []

for ym in sorted(holdout_df_eval['year_month'].unique()):
    month_data = holdout_df_eval[holdout_df_eval['year_month'] == ym]
    
    y_true_month = month_data['target_7d'].values
    y_pred_month = month_data['y_pred'].values
    y_proba_month = month_data['y_pred_proba'].values
    
    # Métricas
    acc = accuracy_score(y_true_month, y_pred_month)
    
    # ROC-AUC (apenas se ambas as classes estão presentes)
    if len(np.unique(y_true_month)) > 1:
        auc_month = roc_auc_score(y_true_month, y_proba_month)
    else:
        auc_month = np.nan
    
    month_metrics.append({
        'year_month': str(ym),
        'n_obs': len(month_data),
        'accuracy': acc,
        'roc_auc': auc_month
    })
    
    print(f"{ym}:")
    print(f"  Observações: {len(month_data):,}")
    print(f"  Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    if not np.isnan(auc_month):
        print(f"  ROC-AUC: {auc_month:.4f}")
    else:
        print(f"  ROC-AUC: N/A (apenas uma classe presente)")
    print()

# Resumo
month_metrics_df = pd.DataFrame(month_metrics)

print("\n📈 Resumo por Mês:")
print(f"\nMelhor Accuracy: {month_metrics_df.loc[month_metrics_df['accuracy'].idxmax(), 'year_month']} ({month_metrics_df['accuracy'].max():.4f})")
print(f"Pior Accuracy: {month_metrics_df.loc[month_metrics_df['accuracy'].idxmin(), 'year_month']} ({month_metrics_df['accuracy'].min():.4f})")

if month_metrics_df['roc_auc'].notna().any():
    valid_auc = month_metrics_df[month_metrics_df['roc_auc'].notna()]
    print(f"\nMelhor ROC-AUC: {valid_auc.loc[valid_auc['roc_auc'].idxmax(), 'year_month']} ({valid_auc['roc_auc'].max():.4f})")
    print(f"Pior ROC-AUC: {valid_auc.loc[valid_auc['roc_auc'].idxmin(), 'year_month']} ({valid_auc['roc_auc'].min():.4f})")

# Estabilidade ao longo do tempo
print(f"\n📊 Estabilidade ao longo dos meses:")
print(f"  Desvio padrão Accuracy: {month_metrics_df['accuracy'].std():.4f}")
if month_metrics_df['roc_auc'].notna().any():
    print(f"  Desvio padrão ROC-AUC: {month_metrics_df['roc_auc'].std():.4f}")

print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("🏆 AVALIAÇÃO TOP 1")
print("=" * 80)

print("\nPara cada data do holdout:")
print("  1. Ranquear os 5 FIIs pela probabilidade prevista")
print("  2. Selecionar o FII ranking 1")
print("  3. Verificar se superou o IFIX nos próximos 7 pregões")
print("  4. Calcular taxa Top 1 de outperformance\n")

# Agrupar por data
dates_in_holdout = sorted(holdout_df_eval['date'].unique())

top1_results = []

for date in dates_in_holdout:
    date_data = holdout_df_eval[holdout_df_eval['date'] == date].copy()
    
    # Ranquear por probabilidade (descendente)
    date_data = date_data.sort_values('y_pred_proba', ascending=False)
    
    # Top 1
    if len(date_data) > 0:
        top1 = date_data.iloc[0]
        
        # Verificar se o Top 1 realmente superou o IFIX
        # target_7d já contém essa informação (True = superou IFIX)
        top1_beat_ifix = top1['target_7d']
        
        # Verificar se foi o melhor retorno absoluto entre os 5 FIIs
        # Para isso, precisamos comparar o target_alpha_7d
        best_alpha = date_data['target_alpha_7d'].max()
        top1_alpha = top1['target_alpha_7d']
        top1_best_return = (top1_alpha == best_alpha)
        
        top1_results.append({
            'date': date,
            'top1_ticker': top1['ticker'],
            'top1_proba': top1['y_pred_proba'],
            'top1_beat_ifix': top1_beat_ifix,
            'top1_best_return': top1_best_return,
            'top1_alpha_7d': top1_alpha,
            'best_alpha_7d': best_alpha
        })

top1_df = pd.DataFrame(top1_results)

# Calcular taxa Top 1
total_dates = len(top1_df)
top1_hits = top1_df['top1_beat_ifix'].sum()
top1_rate = top1_hits / total_dates if total_dates > 0 else 0

top1_best_hits = top1_df['top1_best_return'].sum()
top1_best_rate = top1_best_hits / total_dates if total_dates > 0 else 0

print("\n🎯 Resultados Top 1:\n")
print(f"  Total de datas avaliadas: {total_dates}")
print(f"  Acertos Top 1 (superou IFIX): {top1_hits}")
print(f"  Taxa Top 1: {top1_rate:.4f} ({top1_rate*100:.2f}%)")
print(f"\n  Top 1 foi o melhor retorno: {top1_best_hits}")
print(f"  Taxa Top 1 melhor retorno: {top1_best_rate:.4f} ({top1_best_rate*100:.2f}%)")

print(f"\n📊 Meta aproximada: 58%")
if top1_rate >= 0.58:
    print(f"  ✅ Meta ATINGIDA! ({top1_rate*100:.2f}% >= 58%)")
else:
    diff = (0.58 - top1_rate) * 100
    print(f"  ⚠️  Meta NÃO atingida (faltaram {diff:.2f} pontos percentuais)")

print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("📊 TOP 1 POR MÊS E TICKER")
print("=" * 80)

# Top 1 por mês
top1_df['year_month'] = pd.to_datetime(top1_df['date']).dt.to_period('M')

print("\n📅 Taxa Top 1 por Mês:\n")

top1_month_stats = []

for ym in sorted(top1_df['year_month'].unique()):
    month_data = top1_df[top1_df['year_month'] == ym]
    
    total = len(month_data)
    hits = month_data['top1_beat_ifix'].sum()
    rate = hits / total if total > 0 else 0
    
    top1_month_stats.append({
        'month': str(ym),
        'total': total,
        'hits': hits,
        'rate': rate
    })
    
    print(f"{ym}: {hits}/{total} ({rate*100:.2f}%)")

top1_month_df = pd.DataFrame(top1_month_stats)

print(f"\n📈 Melhor mês: {top1_month_df.loc[top1_month_df['rate'].idxmax(), 'month']} ({top1_month_df['rate'].max()*100:.2f}%)")
print(f"Pior mês: {top1_month_df.loc[top1_month_df['rate'].idxmin(), 'month']} ({top1_month_df['rate'].min()*100:.2f}%)")
print(f"Desvio padrão: {top1_month_df['rate'].std()*100:.2f} pontos percentuais")

# Top 1 por ticker selecionado
print("\n\n🎯 Taxa Top 1 por Ticker Selecionado:\n")

top1_ticker_stats = []

for ticker in sorted(top1_df['top1_ticker'].unique()):
    ticker_data = top1_df[top1_df['top1_ticker'] == ticker]
    
    total = len(ticker_data)
    hits = ticker_data['top1_beat_ifix'].sum()
    rate = hits / total if total > 0 else 0
    
    top1_ticker_stats.append({
        'ticker': ticker,
        'total': total,
        'hits': hits,
        'rate': rate
    })
    
    print(f"{ticker}: {hits}/{total} ({rate*100:.2f}%) - Selecionado {total} vezes")

top1_ticker_df = pd.DataFrame(top1_ticker_stats)

print(f"\n📈 Ticker mais selecionado: {top1_ticker_df.loc[top1_ticker_df['total'].idxmax(), 'ticker']} ({top1_ticker_df['total'].max()} vezes)")
print(f"Ticker com melhor taxa quando selecionado: {top1_ticker_df.loc[top1_ticker_df['rate'].idxmax(), 'ticker']} ({top1_ticker_df['rate'].max()*100:.2f}%)")

print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("📊 VISUALIZAÇÕES")
print("=" * 80)

# Criar figura com subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Distribuição de probabilidades
ax1 = axes[0, 0]
ax1.hist(y_pred_proba, bins=50, alpha=0.7, edgecolor='black')
ax1.axvline(0.5, color='red', linestyle='--', label='Threshold 0.5')
ax1.set_xlabel('Probabilidade Prevista')
ax1.set_ylabel('Frequência')
ax1.set_title('Distribuição de Probabilidades')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. ROC Curve
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_holdout, y_pred_proba)

ax2 = axes[0, 1]
ax2.plot(fpr, tpr, linewidth=2, label=f'ROC (AUC = {roc_auc:.4f})')
ax2.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5000)')
ax2.set_xlabel('Taxa de Falsos Positivos')
ax2.set_ylabel('Taxa de Verdadeiros Positivos')
ax2.set_title('Curva ROC')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Accuracy por mês
ax3 = axes[1, 0]
month_metrics_df_sorted = month_metrics_df.sort_values('year_month')
ax3.plot(range(len(month_metrics_df_sorted)), month_metrics_df_sorted['accuracy'].values, 
         marker='o', linewidth=2, markersize=8)
ax3.axhline(accuracy, color='red', linestyle='--', label=f'Média ({accuracy:.4f})')
ax3.set_xlabel('Mês')
ax3.set_ylabel('Accuracy')
ax3.set_title('Accuracy por Mês')
ax3.set_xticks(range(len(month_metrics_df_sorted)))
ax3.set_xticklabels(month_metrics_df_sorted['year_month'].values, rotation=45)
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Taxa Top 1 por mês
ax4 = axes[1, 1]
top1_month_df_sorted = top1_month_df.sort_values('month')
ax4.plot(range(len(top1_month_df_sorted)), top1_month_df_sorted['rate'].values * 100, 
         marker='o', linewidth=2, markersize=8, color='green')
ax4.axhline(top1_rate * 100, color='red', linestyle='--', label=f'Média ({top1_rate*100:.2f}%)')
ax4.axhline(58, color='orange', linestyle='--', label='Meta (58%)')
ax4.set_xlabel('Mês')
ax4.set_ylabel('Taxa Top 1 (%)')
ax4.set_title('Taxa Top 1 por Mês')
ax4.set_xticks(range(len(top1_month_df_sorted)))
ax4.set_xticklabels(top1_month_df_sorted['month'].values, rotation=45)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Visualizações geradas")
print("\n" + "=" * 80)

In [0]:
print("=" * 80)
print("🏆 RELATÓRIO FINAL - TESTE DE HOLDOUT")
print("=" * 80)

print("\n" + "=" * 80)
print("RESPOSTA ÀS 10 PERGUNTAS CRÍTICAS")
print("=" * 80)

print("\n1️⃣ O modelo generalizou para o período completamente intocado?")
if roc_auc > 0.50:
    print(f"   ✅ SIM. ROC-AUC de {roc_auc:.4f} indica desempenho superior ao acaso.")
else:
    print(f"   ❌ NÃO. ROC-AUC de {roc_auc:.4f} não supera o acaso (0.50).")

print(f"\n2️⃣ O ROC-AUC permaneceu superior a 0.50?")
if roc_auc > 0.50:
    diff_from_random = (roc_auc - 0.50) * 100
    print(f"   ✅ SIM. ROC-AUC = {roc_auc:.4f} ({diff_from_random:.2f} pontos acima do acaso).")
else:
    print(f"   ❌ NÃO. ROC-AUC = {roc_auc:.4f}.")

print(f"\n3️⃣ Qual foi a taxa de acerto geral?")
print(f"   Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   Precision: {precision:.4f}")
print(f"   Recall: {recall:.4f}")
print(f"   F1-Score: {f1:.4f}")

print(f"\n4️⃣ Qual foi a taxa Top 1 de outperformance?")
print(f"   Taxa Top 1: {top1_rate:.4f} ({top1_rate*100:.2f}%)")
print(f"   Acertos: {top1_hits}/{total_dates}")
print(f"   Top 1 foi o melhor retorno: {top1_best_rate:.4f} ({top1_best_rate*100:.2f}%)")

print(f"\n5️⃣ A taxa Top 1 ficou próxima ou acima de 58%?")
if top1_rate >= 0.58:
    print(f"   ✅ SIM. {top1_rate*100:.2f}% >= 58% (meta atingida).")
elif top1_rate >= 0.55:
    diff = (0.58 - top1_rate) * 100
    print(f"   ⚠️  PRÓXIMO. {top1_rate*100:.2f}% (faltaram {diff:.2f} pontos percentuais).")
else:
    diff = (0.58 - top1_rate) * 100
    print(f"   ❌ NÃO. {top1_rate*100:.2f}% (faltaram {diff:.2f} pontos percentuais).")

print(f"\n6️⃣ O desempenho foi estável ao longo dos meses?")
std_acc = month_metrics_df['accuracy'].std()
std_top1 = top1_month_df['rate'].std()
print(f"   Desvio padrão Accuracy: {std_acc:.4f}")
print(f"   Desvio padrão Taxa Top 1: {std_top1:.4f} ({std_top1*100:.2f} p.p.)")
if std_acc < 0.10 and std_top1 < 0.15:
    print(f"   ✅ Estável. Variação aceitável ao longo dos meses.")
else:
    print(f"   ⚠️  Variação considerável. Desempenho oscila entre meses.")

print(f"\n7️⃣ Quais tickers tiveram melhor e pior desempenho?")
print(f"   Melhor Accuracy: {ticker_metrics_df.loc[ticker_metrics_df['accuracy'].idxmax(), 'ticker']} ({ticker_metrics_df['accuracy'].max():.4f})")
print(f"   Pior Accuracy: {ticker_metrics_df.loc[ticker_metrics_df['accuracy'].idxmin(), 'ticker']} ({ticker_metrics_df['accuracy'].min():.4f})")
if ticker_metrics_df['roc_auc'].notna().any():
    valid_auc = ticker_metrics_df[ticker_metrics_df['roc_auc'].notna()]
    print(f"   Melhor ROC-AUC: {valid_auc.loc[valid_auc['roc_auc'].idxmax(), 'ticker']} ({valid_auc['roc_auc'].max():.4f})")
    print(f"   Pior ROC-AUC: {valid_auc.loc[valid_auc['roc_auc'].idxmin(), 'ticker']} ({valid_auc['roc_auc'].min():.4f})")

print(f"\n8️⃣ O resultado confirma ou rejeita a tese principal do projeto?")
print(f"   Tese: É possível prever FIIs que superarão o IFIX em 7 pregões.")
if roc_auc > 0.55 and top1_rate >= 0.52:
    print(f"   ✅ CONFIRMA. O modelo demonstra capacidade preditiva acima do acaso.")
    print(f"   ROC-AUC ({roc_auc:.4f}) e Top 1 ({top1_rate*100:.2f}%) são consistentemente superiores.")
elif roc_auc > 0.50:
    print(f"   ⚠️  CONFIRMA PARCIALMENTE. Há sinal preditivo, mas modesto.")
else:
    print(f"   ❌ REJEITA. Não há evidência robusta de capacidade preditiva.")

print(f"\n9️⃣ O modelo está suficientemente robusto para ser apresentado como resultado final?")
if roc_auc > 0.55 and top1_rate >= 0.52 and std_acc < 0.10:
    print(f"   ✅ SIM. Métricas são estáveis e superiores ao acaso no período intocado.")
    print(f"   O modelo pode ser considerado validado para demonstração.")
elif roc_auc > 0.50:
    print(f"   ⚠️  COM RESSALVAS. O modelo tem sinal preditivo, mas limitado.")
    print(f"   Deve ser apresentado com transparência sobre limitações.")
else:
    print(f"   ❌ NÃO. Desempenho insuficiente para apresentação como modelo validado.")

print(f"\n🔟 Quais limitações devem ser documentadas?")
print(f"\n   Limitações identificadas:")
print(f"   1. Apenas 5 FIIs analisados (amostra pequena)")
print(f"   2. Período de treino: 2020-2025 (inclui pandemia e recuperação)")
print(f"   3. Período de holdout: 2025-2026 (1 ano, pode não capturar ciclos completos)")
print(f"   4. Features macroeconômicas com defasagem de divulgação")
print(f"   5. Não considera custos de transação ou impacto de mercado")
print(f"   6. Backtest é uma simulação; desempenho real pode diferir")
if top1_rate < 0.58:
    print(f"   7. Taxa Top 1 não atingiu a meta de 58%")
if std_top1 > 0.15:
    print(f"   8. Variação considerável na taxa Top 1 entre meses")

print("\n" + "=" * 80)
print("✅ TESTE FINAL DE HOLDOUT CONCLUÍDO")
print("=" * 80)

print(f"\n📊 RESUMO EXECUTIVO:\n")
print(f"  Período: {holdout_start.date()} a {holdout_end.date()}")
print(f"  Observações: {len(holdout_df):,}")
print(f"  ROC-AUC: {roc_auc:.4f}")
print(f"  Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Taxa Top 1: {top1_rate:.4f} ({top1_rate*100:.2f}%)")
print(f"  Meta Top 1: 58%")
print(f"  Status Meta: {'ATINGIDA' if top1_rate >= 0.58 else 'NÃO ATINGIDA'}")

print("\n" + "=" * 80)